In [1]:
import os
import openpyxl
from openpyxl.utils import range_boundaries

def clean_excel_files(source_folder, dest_folder):
    for root, dirs, files in os.walk(source_folder):
        for filename in files:
            if filename.endswith('.xlsx') and not filename.startswith('~$'):
                input_file_path = os.path.join(root, filename)
                relative_path = os.path.relpath(root, source_folder)
                output_dir_path = os.path.join(dest_folder, relative_path)
                output_file_path = os.path.join(output_dir_path, filename)

                if not os.path.exists(output_dir_path):
                    os.makedirs(output_dir_path)

                print(f"Processing: {input_file_path}")

                try:
                    wb = openpyxl.load_workbook(input_file_path, data_only=False)
                    sheet_modified = False

                    for sheet_name in wb.sheetnames:
                        # ROA setup
                        scanned_rows = set()
                        sheet = wb[sheet_name]

                        max_row_scan = min(6, sheet.max_row)

                        for row_idx in range(1, max_row_scan + 1):
                            for cell in sheet[row_idx]:
                                if cell.value and "ROA" in str(cell.value).upper():
                                    scanned_rows.add(cell.column)
                        
                        scanned_columns = list(scanned_rows)
                        if not scanned_columns:
                            print(f"  - No 'ROA' columns found in sheet '{sheet_name}' (rows 1-6)")
                            continue

                        print(f"  - Found ROA columns in sheet '{sheet_name}': {scanned_columns}")
                        start_row = 7

                        rows_to_remove = []

                        for row_idx in range(sheet.max_row, start_row - 1, -1):
                            # row_data = list(sheet.iter_rows(min_row=row_idx, max_row=row_idx, values_only=True))[0]
                            remove_this_row = False

                            for col_idx in scanned_columns:
                                cell_getter = sheet.cell(row=row_idx, column=col_idx)
                                cell_val = cell_getter.value

                                if cell_val is None:
                                    continue
                                if isinstance(cell_val, (int, float)) and cell_val == 0:
                                    remove_this_row = True
                                    break
                                if isinstance(cell_val, str) and cell_val.strip().upper() == "NA":
                                    remove_this_row = True
                                    break

                            if remove_this_row:
                                rows_to_remove.append(row_idx)

                        if rows_to_remove:
                            rows_to_remove.sort(reverse=True)
                            for row_idx in rows_to_remove:
                                sheet.delete_rows(row_idx, 1)
                            sheet_modified = True
                            print(f"  - Removed {len(rows_to_remove)} rows from sheet '{sheet_name}'")

                    wb.save(output_file_path)
                    if sheet_modified:
                        print(f"  -> Saved cleaned file to: {output_file_path}")
                    else:
                        print(f"  -> No changes needed. Saved copy to: {output_file_path}")

                except Exception as e:
                    print(f"Error processing file {filename}: {e}")

source_directory = "E:\\hneu\\data2"
destination_directory = "E:\\hneu\\data4"

if __name__ == "__main__":
    if os.path.exists(source_directory):
        clean_excel_files(source_directory, destination_directory)
        print("\nAll files processed successfully.")
    else:
        print(f"Error: Source directory '{source_directory}' does not exist.")


Processing: E:\hneu\data2\bank20142017\Argentina.xlsx
  - Found ROA columns in sheet 'Sheet1': [38, 39, 40, 41, 42, 43, 44, 45]
  - No 'ROA' columns found in sheet 'Screening Criteria' (rows 1-6)
  -> No changes needed. Saved copy to: E:\hneu\data4\bank20142017\Argentina.xlsx
Processing: E:\hneu\data2\bank20142017\Australia.xlsx
  - Found ROA columns in sheet 'Sheet1': [38, 39, 40, 41, 42, 43, 44, 45]
  - Removed 1 rows from sheet 'Sheet1'
  - No 'ROA' columns found in sheet 'Screening Criteria' (rows 1-6)
  -> Saved cleaned file to: E:\hneu\data4\bank20142017\Australia.xlsx
Processing: E:\hneu\data2\bank20142017\Austria.xlsx
  - Found ROA columns in sheet 'Sheet1': [38, 39, 40, 41, 42, 43, 44, 45]
  - Removed 1 rows from sheet 'Sheet1'
  - No 'ROA' columns found in sheet 'Screening Criteria' (rows 1-6)
  -> Saved cleaned file to: E:\hneu\data4\bank20142017\Austria.xlsx
Processing: E:\hneu\data2\bank20142017\Brazil.xlsx
  - Found ROA columns in sheet 'Sheet1': [38, 39, 40, 41, 42, 43, 